# Работа с файлами в Python.Обработка ошибок. Try/except

*По материалам Дарьи Касьяненко*

## Файловый ввод-вывод

Как правило, если указать в Python не полный путь к файлу, а только его название, то он будет искать файл в рабочей директории. Как узнать, где это?


In [ ]:
import os
os.getcwd()

'/content'

Функция getcwd() из модуля os возвращает нам путь к вашей рабочей папке. Так, например, в Windows по умолчанию Anaconda делает рабочей папкой для Jupyter папку пользователя в Users. Это можно изменить или глобально, прописав путь к вашей папке в свойствах, или локально в рамках сессии.

Функция chdir() принимает в качестве аргумента путь к папке и меняет рабочую директорию. Теперь к файлам, хранящимся в ней, вы сможете обращаться без полного пути. Также все новые файлы будут сохраняться туда же.

In [ ]:
os.chdir('/content')
# для Windows такая запись: 'C:\\Users\\username\\HSE\\new_folder'
os.getcwd()

'/content'

.listdir() вернет нам список содержимого директории. Очень полезная функция - можно запустить цикл, если нужно обработать все файлы в папке.

In [ ]:
os.listdir()

['.config', 'mbox.txt', 'sample.xlsx', 'sample_data']

Давайте попробуем создать файл, записать в него что-нибудь и сохранить.

In [ ]:
f = open("test.txt", 'w', encoding='utf-8')

Функция open() возвращает файловый объект и мы используем ее обычно с двумя аргументами - имя файла и режим (например, запись или чтение). Выше мы открыли файл test.txt в режиме записи 'w' (если такого файла не существовало, он будет создан).

Такой объект называется file handle или дескриптор файла.

![](https://www.py4e.com/images/handle.svg)  
Source: https://www.py4e.com/html3/07-files

Какие могут быть режимы открытия файла (mode):

* 'r' - read, только чтение
* 'w' - write, только запись (если файл с таким именем существовал, он будет удален).
* 'a' - append, новые данные будут записаны в конец файла
* 'r+' - чтение+запись.

Если не передать второй аргумент, то файл автоматически откроется в режиме чтения.

Encoding - именнованный параметр, если работаете с кириллицей или языками со спецсимволами, то лучше задать utf-8.

In [ ]:
f.write('Hello, world!')
f.close()

Метод write записал данные в наш файл. После этого файл нужно закрыть, чтобы он выгрузился из оперативной памяти. Если этого не сделать, то в какой-то момент питоновский сборщик мусора все равно до него доберется и закроет файл, но большие файлы могут съедать достаточно много ресурсов, поэтому лучше за этим следить.

Теперь дававайте попробуем открыть в режиме чтения.

In [ ]:
f = open("test.txt", 'r', encoding='utf8')
print(f.read())
f.close()

Hello, world!


После того, как мы закрыли файл, обратиться к нему больше нельзя.

In [ ]:
f.read()

ValueError: I/O operation on closed file.

Хорошим тоном при работе с файлами считается открывать их с помощью ключевого слова with. Преимущество этого способа в том, что файл закроется автоматически, когда закончатся вложенные операции.

In [ ]:
with open('test.txt') as f: # открыли файл, не указали режим, по умолчанию - чтение
    read_data = f.read() # считали данные из файла в переменную
# операции закончились, файл сам закрылся

In [ ]:
print(read_data)

Hello, world!


А теперь попробуем записать в файл новые строки.

In [ ]:
with open('test.txt', 'a') as f: # открыли файл
    f.write('\nЭто новая строка!\nА эта еще новее!') # считали данные из файла в переменную
# операции закончились, файл сам закрылся

Еще один вариант записать данные в файл вот так:

In [ ]:
with open('test.txt', 'a') as f:
    print("\nЭто последняя строка", file=f)

In [ ]:
with open('test.txt') as f:
    text = f.read()

In [ ]:
text

'Hello, world!\nПривет\nТретья строка\nЧетверта строка'

In [ ]:
print(text)

Hello, world!
Привет
Третья строка
Четверта строка


Мы выше уже видели два метода файла .write() и .read(). Еще один метод, который очень часто используется - это readline. Он позволяет не загружать файл целиком в память, а считывать его построчно. Знаком остановки здесь будет выступать \n

In [ ]:
f = open('test.txt', 'r')
f.readline()

'Hello, world!'

In [ ]:
f.readline()

''

.readline() - генератор. При обращении он выдает нам новую строку.

In [ ]:
f.readline() # вызвали третий раз

''

In [ ]:
f.close()

Также, чтобы прочитать все строки поочередно, можно запустить цикл. Тут не стоит забывать, что переменная f, хоть и прикидывается списком строк, когда мы её итерируем, на самом деле таковым не является. В действительности при открытии файла мы запоминаем позицию, на которой мы этот файл читаем. Изначально она указывает на самое начало файла, но с каждой итерацией сдвигается. Когда мы прочитаем файл целиком, дальнейшие попытки из него что-то прочитать ни к чему не приведут: указатель текущей позиции сдвинулся до самого конца и файл закончился.

In [ ]:
f = open('test.txt', 'r')

for line in f:
    print(line, end='')

for line in f: # обратите внимание, что этот цикл не выполняется
    print(line, end='')

f.close()

Hello, world!
Привет
Третья строка
Четверта строка

Файл можно перемотать на начало, если воспользоваться методом .seek(), который возвращается к символу на этой позиции.

In [ ]:
f = open('test.txt', 'r')

for line in f:
    print(line, end='')

f.seek(0) # вернули файл на начало

for line in f: # теперь печатает!
    print(line, end='')
f.close()

Hello, world!
Привет
Третья строка
Четверта строкаHello, world!
Привет
Третья строка
Четверта строка

Если методу read() передать целое число, то питон прочитает только заданное количество символов или битов, если информация в файле записана в бинарном формате.

In [ ]:
with open('test.txt') as f:
    print(f.read(6))

Hello,


Чтобы считать все строки файла в список, можно вызвать список от файлового объекта или использовать метод .readlines().

In [ ]:
f = open('test.txt', 'r')
print(list(f))
f.close()

['Hello, world!\n', 'Это новая строка!\n', 'А эта еще новее!\n', 'Это последняя строка\n', '\n', 'Это последняя строка\n']


In [ ]:
f = open('test.txt', 'r')
print(f.readlines())
f.close()

['Hello, world!\n', 'Это новая строка!\n', 'А эта еще новее!\n', 'Это последняя строка\n', '\n', 'Это последняя строка\n']


## Try except

In [ ]:
try:
    a = int(input())
    b = int(input())
    print(a / b)
except ZeroDivisionError:
    print('Не делите на ноль')
except:
    print('Что-то не так, не знаю что')
print('Продолжаем...')

павпа
Что-то не так, не знаю что
Продолжаем...


## Немного про исключения

Почти наверняка вы уже делали что-то, что приводило к сообщению об ошибке. Сегодня мы научимся их обрабатывать и писать собственные исключения - наши инструкции для Python, чтобы код не ломался, как только что-то пойдет не так.

Например, мы хотим открывать файл по запросу от пользователя, но хотим обработать случай, когда пользователь введет неправильное название.

In [ ]:
fname = input('Введите название файла: ')
fhand = open(fname) # введем название несуществующего файла

Введите название файла: abfgfd


FileNotFoundError: [Errno 2] No such file or directory: 'abfgfd'

Напишем блок try/except. Try будет исполняться до тех пор, пока что-то не сломается. Как только возникнет ошибка, ваша программа перейдет в часть except и выполнит действие, описанное в ней. Сообщения об ошибке выведено не будет.

In [ ]:
fname = input('Введите название файла: ') # передадим валидное название файла

try:
    fhand = open(fname)
    count = 0
    for line in fhand:
        if line.startswith('From '):
            count = count + 1
    fhand.close()
    print(f'В файле {fname} всего {count} строк, начинающихся с From')

except:
    print('Файл невозможно открыть:', fname)

print('Программа работает')


Введите название файла: ahhytryjrtd
Файл невозможно открыть: ahhytryjrtd
Программа работает


Except позволил нам избежать ошибки и остановки работы программы

Естественно, try-except можно использовать не только с файлами.

In [ ]:
try:
    print(y) # переменной y пока не существует
except NameError:
    print("An exception occurred") # нет ошибки

An exception occurred


In [ ]:
print(y)

NameError: name 'y' is not defined

Но что делать, если мы хотим пропускать только определенный вид ошибок, но видеть сообщения об остальных? Try/except хороший инструмент для отладки кода. Давайте посмотрим, как называется ошибка при попытке вызова неопределенной переменной.

In [ ]:
print(y)

NameError: name 'y' is not defined

`NameError: name 'y' is not defined` - название нашей ошибки NameError, давайте обрабатывать его отдельно.

In [ ]:
try:
    print(y)
except NameError:
    print("Variable y is not defined")
except:
    print("Something else went wrong")

Variable y is not defined


А теперь давайте попробуем разделить что-нибудь на 0.

In [ ]:
try:
    print(10/0)
except NameError:
    print("Variable y is not defined")
except:
    print("Something else went wrong")

Something else went wrong


Если после вывода в except использовать ключевое слово raise, то программа закончит работу ошибкой. Отличие от того, чтобы просто не использовать except для других случаев в том, что до окончания работы программы будет выпонелнено все, что написано внутри except. Сравним.

In [ ]:
try:
    print(10/0)
except NameError:
    print("Variable y is not defined")
except:
    print("Something else went wrong")
    raise

Something else went wrong


ZeroDivisionError: division by zero

Except может одновременно обрабатывать несколько исключений.

In [ ]:
a

7

In [ ]:
try:
    x = int(input())
    y = int(input())
    b = input()
    print(x / y, x + b)
except (NameError, ZeroDivisionError):
    print("Variable y is not defined or someone tried to divide by zero")
except TypeError:
    print('Что-то не так с типами данных')
except:
    print("Something else went wrong")

7
2
1
Что-то не так с типами данных


Except может использоваться с else - будет исполнено, если не было ошибки.

In [ ]:
try:
    print(10)
except NameError:
    print("Variable y is not defined")
except:
    print("Something else went wrong")
else:
    print('Nothing went wrong')

10
Nothing went wrong


Еще один важный момент - использование ключевого слова pass. Достаточно часто пригождается при вэб-скрейпинге. Так вы можете заложить воможность того, что данные не придут в нужном формате и, например, не записывать их в ячейку таблицы, а просто пропустить.

In [ ]:
try:
    print(10/0)
except NameError:
    print("Variable y is not defined")
except:
    pass # ничего не происходит, но и ошибки нет

In [ ]:
for i in range(1, 10):
    for j in range(1, 10):
        print(f'{i} * {j} = {i * j}')

In [ ]:
x = -1

if x < 0:
    raise Exception("Sorry, no numbers below zero")

Exception: Sorry, no numbers below zero

Или же вы можете использовать уже существующие исключения и персонализировать сообщение об ошибке.

In [ ]:
x = "hello"

if not type(x) is int:
    raise TypeError("Only integers are allowed")

TypeError: Only integers are allowed

Давайте попробуем решить такую задачу: вы ждете от пользователя на ввод число. Будем запрашивать число до тех пор, пока не добьемся правильного ввода.

In [ ]:
while True:
    try:
        x = int(input("Введите число: "))
        break
    except ValueError:
        print("Это было не число. Давайте еще раз.")

Введите число: abc
Это было не число. Давайте еще раз.
Введите число: bdf
Это было не число. Давайте еще раз.
Введите число: 7


# Работа с датой и временем

https://docs.python.org/3/library/datetime.html#module-datetime

Модуль datetime предоставляет классы для работы с датами и временем

In [ ]:
import datetime

Модуль `datetime` предоставляет классы для работы с датами и временем, позволяя выполнять различные операции, такие как создание объектов даты и времени, их форматирование и преобразование.

### Основные классы модуля `datetime`

1. **`datetime.date`**:
   - Представляет дату (год, месяц, день).
   - Создание объекта:
   - Атрибуты: `year`, `month`, `day`.

In [ ]:
import datetime
d = datetime.date(2022, 12, 25)
print(d)

2022-12-25


In [ ]:
type(d)

datetime.date

In [ ]:
d.year

2022

In [ ]:
d.month

12

In [ ]:
d.day

25


2. **`datetime.time`**:
   - Представляет время (час, минуты, секунды).
   - Создание объекта:
   - Атрибуты: `hour`, `minute`, `second`, `microsecond`, `tzinfo`.

In [ ]:
t = datetime.time(8, 48, 45)
print(t)

08:48:45


In [ ]:
t.second

45

In [ ]:
t.tzinfo


3. **`datetime.datetime`**:
   - Содержит информацию как о дате, так и о времени.
  

In [ ]:
dt = datetime.datetime(2022, 12, 25, 15, 30)
print(dt)

2022-12-25 15:30:00


In [ ]:
dt.year

2022

In [ ]:
dt.hour

15


4. **`datetime.timedelta`**:
   - Представляет разницу между двумя моментами времени.


In [ ]:
delta = datetime.timedelta(days=5)
print(delta)

5 days, 0:00:00



5. **`datetime.tzinfo`** и **`datetime.timezone`**:
   - Используются для работы с часовыми поясами.

In [ ]:

# Создаем объект временной зоны с смещением +3 часа от UTC
offset = datetime.timedelta(hours=3)
tz = datetime.timezone(offset, name='МСК')

# Получаем текущее время с учетом временной зоны
dt = datetime.datetime.now(tz)
print(dt)  # Вывод: текущее время с учетом МСК



2025-04-07 21:37:18.645560+03:00



### Получение текущей даты и времени

Для получения текущей даты и времени можно использовать метод `now()`:

In [ ]:
current_datetime = datetime.datetime.now()
print(current_datetime)

2025-04-07 18:37:46.599555


In [ ]:
# Получаем текущую дату
current_date = datetime.date.today()
print(current_date)

2025-04-07


Операции с датами:

In [ ]:
# Добавляем 10 дней к текущей дате
future_date = current_date + datetime.timedelta(days=50)
print(future_date)

2025-05-27


In [ ]:
d1 = datetime.date(2024, 12, 1)
d2 = datetime.date(2024, 12, 5)


In [ ]:
d1 > d2

False

Преобразование строки с датой:

In [ ]:
date_string = "2025-04-07"
date_object = datetime.datetime.strptime(date_string, "%Y-%m-%d").date()
print(date_object) # Вывод: 2025-04-07

2025-04-07


Форматирование даты в строку:

In [ ]:
formatted_date = current_date.strftime("%d-%m-%Y")
print(formatted_date)  # Вывод: "05-12-2024"

07-04-2025


In [ ]:
type(formatted_date)

str

%Y: год с четырьмя цифрами (например, 2024).  
%y: год с двумя цифрами (например, 24 для 2024).  
%m: месяц с нулевым дополнением (01-12).  
%d: день месяца с нулевым дополнением (01-31).  
%H: часы в 24-часовом формате (00-23).  
%M: минуты с нулевым дополнением (00-59).  
%S: секунды с нулевым дополнением (00-59).  
%B: полное название месяца  
%b: сокращенное название месяца  

In [ ]:
datetime.datetime.strptime(date_string, "%Y-%m-%d")

datetime.datetime(2025, 4, 7, 0, 0)

In [ ]:
current_date.strftime("%B-%d-%m-%y-%H-%M-%S")

'April-07-04-25-00-00-00'